# 3x Steane Code CNOT Logical Error Rates

This notebook benchmarks the logical CNOT experiment using three [[7,1,3]] Steane codes (one for control, one for target, one as ancilla) with ancilla-mediated homological-measurement construction. Visualization is intentionally skipped; only logical-rate data is saved.

In [ ]:
import importlib
import numpy as np
import sinter
import stim
from ldpc import SinterBpOsdDecoder

from pathlib import Path

from epic import (
    AllocCode,
    FreeCode,
    InitCode,
    PauliChar,
    PauliEigenState,
    PauliString,
    QECCompiler,
    ReadoutCode,
    StimLikeNoiseModel,
)
from epic.modules.qec_gadgets.pauli_product_measurement.homological_measurement import HomologicalMeasurement
from epic.modules.stabilizers_codes.css_code import CSSCode
from epic.modules.stabilizers_codes.rotated_surface_code import RotatedSurfaceCode

base_dir = Path("/Users/lenny/Documents/foo/EPIC-QEC/docs/example_notebooks/cnot_benchmark_result")
logical_rates_dir = base_dir / "logical_rates"
logical_rates_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Define the [[7,1,3]] Steane code
# Parity check matrix for Steane code
H_steane = np.array(
    [
        [0, 0, 0, 1, 1, 1, 1],
        [0, 1, 1, 0, 0, 1, 1],
        [1, 0, 1, 0, 1, 0, 1],
    ],
    dtype=np.uint8,
)

# Logical operators for Steane code
lX = [1, 1, 1, 0, 0, 0, 0]
lZ = [1, 1, 1, 0, 0, 0, 0]

# Create three Steane codes: control, target, and ancilla
control_code = CSSCode.from_css_pcm(
    code_name="steane_ctrl",
    hx=H_steane,
    hz=H_steane,
    logical_qubits=[(lX, lZ)],
)

target_code = CSSCode.from_css_pcm(
    code_name="steane_target",
    hx=H_steane,
    hz=H_steane,
    logical_qubits=[(lX, lZ)],
)

ancilla_code = RotatedSurfaceCode.from_distance(distance=3, code_name="ancilla_d3_steane", system_coordinate=(1, 0))

print(f"Control code: {control_code.name}, n={control_code.n}, k={control_code.k}")
print(f"Target code: {target_code.name}, n={target_code.n}, k={target_code.k}")
print(f"Ancilla code: {ancilla_code.name}, n={ancilla_code.n}, k={ancilla_code.k}")

In [ ]:
PRIMITIVES_CONFIG = {
    "epic.core.qec_primitives.interfaces.ApplyGate": "epic.modules.qec_primitives.apply_gates.simple_gate_application.SimpleGateApplication",
    "epic.core.qec_primitives.interfaces.Readout": "epic.modules.qec_primitives.readouts.naive_readout.NaiveReadout",
    "epic.core.qec_primitives.interfaces.ExtractSyndrome": "epic.modules.qec_primitives.syndrome_extraction.zxcoloring_extraction.ZXColoringExtraction",
}


def compile_steane_cnot_experiment():
    """Compile CNOT experiment with 3 Steane codes."""
    ctrl_lq_names = ["ctrl_lq_0"]
    target_lq_names = ["target_lq_0"]
    anc_lq_names = ["anc_lq_0"]

    program = [
        AllocCode(target_code=control_code, code_varname="ctrl_patch", logical_qubits_varnames=ctrl_lq_names),
        AllocCode(target_code=target_code, code_varname="target_patch", logical_qubits_varnames=target_lq_names),
        AllocCode(target_code=ancilla_code, code_varname="anc_patch", logical_qubits_varnames=anc_lq_names),
        InitCode(targets=["ctrl_patch"], initial_state=PauliEigenState.Z_plus, tag="init_ctrl"),
        InitCode(targets=["target_patch"], initial_state=PauliEigenState.Z_plus, tag="init_target"),
        InitCode(targets=["anc_patch"], initial_state=PauliEigenState.X_plus, tag="init_anc"),
        HomologicalMeasurement(
            targets=["ctrl_lq_0", "anc_lq_0"],
            product_to_measure=PauliString(string=(PauliChar.Z, PauliChar.Z)),
            tag="MZZ_ctrl_anc",
        ),
        HomologicalMeasurement(
            targets=["anc_lq_0", "target_lq_0"],
            product_to_measure=PauliString(string=(PauliChar.X, PauliChar.X)),
            tag="MXX_anc_target",
        ),
        ReadoutCode(targets=["ctrl_patch"], tag="ro_ctrl"),
        ReadoutCode(targets=["target_patch"], tag="ro_target"),
        ReadoutCode(targets=["anc_patch"], tag="ro_anc"),
        FreeCode(code_varname="ctrl_patch"),
        FreeCode(code_varname="target_patch"),
        FreeCode(code_varname="anc_patch"),
    ]

    compiler = QECCompiler(
        config={
            "objective_distance": 3,
            "primitives": PRIMITIVES_CONFIG,
        }
    )
    compiled_program = compiler.compile(program, show_progress=False)

    # Follow the same feedforward convention as in hgp58_16_3.ipynb,
    # but discover exact observable tags emitted by the compiler.
    emitted_tags = [ob.tag for ob in compiled_program.observables]
    readout_tags = [tag for tag in emitted_tags if tag.startswith("readout_")]

    def logical_index(tag: str) -> int:
        # Extract index from tags like "readout_X_lq_0_ro_ctrl"
        if "lq_" in tag:
            parts = tag.split("lq_")
            if len(parts) > 1 and parts[1][0].isdigit():
                return int(parts[1][0])
        return -1

    ancilla_readouts = [tag for tag in readout_tags if "anc" in tag.lower()]
    data_readouts = [tag for tag in readout_tags if "anc" not in tag.lower()]

    control_candidates = [tag for tag in data_readouts if "ctrl" in tag.lower()]
    target_candidates = [tag for tag in data_readouts if "target" in tag.lower()]
    ancilla_candidates = [tag for tag in ancilla_readouts if logical_index(tag) == 0]

    mzz_candidates = [tag for tag in emitted_tags if "MZZ" in tag]
    mxx_candidates = [tag for tag in emitted_tags if "MXX" in tag]

    if len(control_candidates) != 1 or len(target_candidates) != 1:
        raise ValueError(
            "Could not uniquely identify control/target readout tags. "
            f"Control candidates={control_candidates}, target candidates={target_candidates}"
        )
    if len(ancilla_candidates) != 1:
        raise ValueError(f"Could not uniquely identify ancilla readout tag: {ancilla_candidates}")
    if len(mzz_candidates) != 1 or len(mxx_candidates) != 1:
        raise ValueError(
            f"Could not uniquely identify MZZ/MXX observable tags: MZZ={mzz_candidates}, MXX={mxx_candidates}"
        )

    control_tag = control_candidates[0]
    target_tag = target_candidates[0]
    ancilla_tag = ancilla_candidates[0]
    mzz_tag = mzz_candidates[0]
    mxx_tag = mxx_candidates[0]

    stim_observables = [
        [control_tag],
        [mzz_tag, ancilla_tag, target_tag],
    ]

    return compiled_program, stim_observables


print("Compiling 3x Steane CNOT experiment...")
compiled_program, stim_observables = compile_steane_cnot_experiment()
print(f"Compilation successful. Observables: {len(stim_observables)}")

In [ ]:
noise_values = np.logspace(-5, -1, 9)
shots = 100

tasks = []
for physical_noise in noise_values:
    noise_model = StimLikeNoiseModel.from_stim_like_probabilities(
        after_clifford_depolarization=float(physical_noise),
        after_reset_flip_probability=float(physical_noise),
        before_measure_flip_probability=float(physical_noise),
        before_round_data_depolarization=float(physical_noise),
    )
    stim_program = compiled_program.to_stim_program(stim_observables, noise_model=noise_model)
    tasks.append(
        sinter.Task(
            circuit=stim.Circuit(stim_program),
            json_metadata={"d": 3, "p": float(physical_noise), "code": "3x_steane"},
        )
    )

print("Starting BPOSD benchmark run")
print(f"  shots={shots}, noise_points={len(noise_values)}, decoder=bposd")


def log_progress(progress: sinter.Progress) -> None:
    if progress.status_message:
        print(f"[bposd] {progress.status_message}")


collected_stats = sinter.collect(
    num_workers=4,
    tasks=tasks,
    decoders=["bposd"],
    custom_decoders={
        "bposd": SinterBpOsdDecoder(
            max_iter=0,
            bp_method="ms",
            ms_scaling_factor=0.625,
            osd_method="osd0",
            osd_order=0,
        )
    },
    max_shots=shots,
    progress_callback=log_progress,
    print_progress=True,
)

stats_by_noise = {float(stats.json_metadata["p"]): stats for stats in collected_stats}
logical_error_rates = np.array(
    [
        stats_by_noise[float(physical_noise)].errors / stats_by_noise[float(physical_noise)].shots
        for physical_noise in noise_values
    ]
)

result_title = "EPIC 3x Steane CNOT logical error rate"
result_description = (
    "Logical failure probability for ancilla-mediated CNOT on 3x [[7,1,3]] Steane codes "
    "(one for control, one for target, one as ancilla), using HM-based ZZ/XX product measurements "
    "with feedforward observable definition. A shot is counted as a logical fail when any tracked observable is 1. "
    "Noise uses EPIC StimLikeNoiseModel with matched p for reset flip, Clifford depolarization, "
    "pre-measurement flip, and round-data depolarization channels."
)

results_path = logical_rates_dir / "steane_cnot_rates.npz"
np.savez_compressed(
    results_path,
    distance=np.array([3], dtype=np.int64),
    noise_values=noise_values,
    logical_error_rates=logical_error_rates,
    shots=np.array([shots], dtype=np.int64),
    code=np.array(["3x_steane"]),
    title=np.array([result_title]),
    description=np.array([result_description]),
)

print(f"\nSaved 3x Steane CNOT rates to: {results_path.name}")
for physical_noise, logical_error_rate in zip(noise_values, logical_error_rates):
    print(f"  p={physical_noise:.2e} -> P_L={logical_error_rate:.6f}")